# Illustrated Tensors

This notebook consolidates the tensor-focused illustrated examples: dimension roles, construction, elementwise math, activations, reductions, softmax, shape changes, and selection.

The recurring rule is:

```txt
Find the operation dimensions.
Everything else is parallel.
The same small operation runs independently at every parallel position.
```

How to use this notebook:

1. Read the intuition first.
2. Pause at each **Before you run** prompt and predict the shape or one value.
3. Run the cell.
4. Read the output explanation and connect it back to the dim roles.

The point is not to memorize formulas. The point is to build the habit of reading tensor code by asking: which axes are parallel, which axes participate, which axes disappear, and which axes are produced?


In [ ]:
import { tensor, empty, zeros, ones, full, rand, randn, seed, arange, linspace, cast, move, contiguous, finite_summary, finite_abs_max, add, sub, mul, div, neg, abs, sign, exp, log, sqrt, square, clamp, where, gt_scalar, masked_fill, relu, sigmoid, tanh, gelu, silu, matmul, mean, sum, max, min, variance, std, argmax, argmin, softmax, reshape, transpose, permute, squeeze, unsqueeze, cat, stack, slice, range, gather, index_select, topk, one_hot } from 'affon:compute'


## Tensor Dim Roles



## 1. Batched matmul

A 3D matmul can be read as several 2D matmuls.

<svg viewBox="0 0 760 160" width="760" height="160" xmlns="http://www.w3.org/2000/svg">
  <style>.t{font:14px monospace}.h{font:600 15px sans-serif}.p{fill:#dbeafe;stroke:#60a5fa}.o{fill:#dcfce7;stroke:#4ade80}.q{fill:#ffedd5;stroke:#fb923c}.g{fill:#f8fafc;stroke:#94a3b8}</style>
  <text class="h" x="20" y="24">[batch, m, k] @ [k, n] -> [batch, m, n]</text>
  <rect class="p" x="30" y="50" width="90" height="36" rx="4"/><text class="t" x="48" y="73">batch</text>
  <rect class="o" x="125" y="50" width="48" height="36" rx="4"/><text class="t" x="144" y="73">m</text>
  <rect class="o" x="178" y="50" width="48" height="36" rx="4"/><text class="t" x="197" y="73">k</text>
  <text class="t" x="245" y="73">@</text>
  <rect class="o" x="275" y="50" width="48" height="36" rx="4"/><text class="t" x="294" y="73">k</text>
  <rect class="q" x="328" y="50" width="48" height="36" rx="4"/><text class="t" x="347" y="73">n</text>
  <text class="t" x="395" y="73">=</text>
  <rect class="p" x="430" y="50" width="90" height="36" rx="4"/><text class="t" x="448" y="73">batch</text>
  <rect class="o" x="525" y="50" width="48" height="36" rx="4"/><text class="t" x="544" y="73">m</text>
  <rect class="q" x="578" y="50" width="48" height="36" rx="4"/><text class="t" x="597" y="73">n</text>
  <text class="t" x="30" y="122">For every batch item, run one ordinary matrix multiply.</text>
</svg>

Here `batch` is parallel. The `m`, `k`, and `n` axes are the matrix-multiply axes.


### Why This Exists

Matrix multiply is the workhorse behind projections, attention, and many neural-network layers. In high-rank tensors, it usually means many ordinary matrix multiplies running side by side.

### Before You Run

For `x: [2, 3, 4]` and `w: [4, 2]`, predict the output shape before running the code.

Ask yourself:

- Which dimension is parallel?
- Which dimension is consumed by the multiply?
- Which new dimension appears?


One output vector is computed from one input row:

```txt
x[0, 0] = [1, 2, 3, 4]

first output  = 1*1 + 2*0 + 3*1 + 4*2  = 12
second output = 1*0 + 2*1 + 3*1 + 4*(-1) = 1

matmul(x, w)[0, 0] = [12, 1]
```

The same rule is applied independently at every `(batch, m)` position.


## 2. Linear projection on a high-rank tensor

A `Linear(in, out)` style projection keeps every leading dimension parallel.

```txt
x:      [batch, time, in_features]
weight: [in_features, out_features]
bias:   [out_features]
y:      [batch, time, out_features]
```

Read it as:

```txt
For every (batch, time) position,
apply the same projection from in_features to out_features.
```


### Why This Exists

A Linear layer changes feature size without mixing the leading positions. That is why the same layer can be applied to every token in a sequence.

### Before You Run

If `x` is shaped `[batch=2, time=3, in_features=4]` and `weight` is shaped `[4, 2]`, predict which axes survive unchanged.


## 3. Reduction dims

A reduction removes or shrinks the dimension it works over.

<svg viewBox="0 0 720 150" width="720" height="150" xmlns="http://www.w3.org/2000/svg">
  <style>.t{font:14px monospace}.h{font:600 15px sans-serif}.p{fill:#dbeafe;stroke:#60a5fa}.r{fill:#fee2e2;stroke:#f87171}.g{fill:#f8fafc;stroke:#94a3b8}</style>
  <text class="h" x="20" y="24">mean(x, dim=2)</text>
  <rect class="p" x="30" y="50" width="90" height="36" rx="4"/><text class="t" x="48" y="73">batch</text>
  <rect class="p" x="125" y="50" width="70" height="36" rx="4"/><text class="t" x="143" y="73">time</text>
  <rect class="r" x="200" y="50" width="90" height="36" rx="4"/><text class="t" x="212" y="73">hidden</text>
  <text class="t" x="315" y="73">-></text>
  <rect class="p" x="365" y="50" width="90" height="36" rx="4"/><text class="t" x="383" y="73">batch</text>
  <rect class="p" x="460" y="50" width="70" height="36" rx="4"/><text class="t" x="478" y="73">time</text>
  <text class="t" x="30" y="122">For every (batch, time), average across hidden.</text>
</svg>


### Why This Exists

Reductions turn many values into fewer values. They are used for summaries, losses, normalization statistics, and metrics.

### Before You Run

For `mean(x, 2)`, decide which axis disappears. Then predict the output shape.


With `keep=true`, the reduced axis stays as size `1`:

```txt
[batch, time, hidden] -> [batch, time, 1]
```

That is useful when you want the result to broadcast back against the original tensor.


## 4. Softmax dims

`softmax(scores, dim)` normalizes along one dimension and keeps the shape unchanged.

For attention-like scores:

```txt
scores: [batch, query_pos, key_pos]
softmax(scores, 2)
```

Read it as:

```txt
For every (batch, query_pos),
make one probability distribution across key_pos.
```


### Why This Exists

Softmax turns raw scores along one axis into a distribution-like set of values. In attention, each query position gets its own distribution over key positions.

### Before You Run

For `scores: [batch=2, query_pos=2, key_pos=3]`, predict the shape after `softmax(scores, 2)`. Does the normalized axis disappear or stay?


## Reusable reading rule

When reading a high-rank op:

1. Find the dimensions the operation actually uses.
2. Treat the other dimensions as parallel positions.
3. Ask which dimensions are preserved, reduced, or produced.

Examples:

| Expression | Parallel dims | Operation dim(s) | Result |
| --- | --- | --- | --- |
| `matmul([B, M, K], [K, N])` | `B`, `M` positions | `K -> N` projection | `[B, M, N]` |
| `mean([B, T, H], 2)` | `B`, `T` | reduce `H` | `[B, T]` |
| `mean([B, T, H], 2, true)` | `B`, `T` | reduce `H` but keep axis | `[B, T, 1]` |
| `softmax([B, Q, K], 2)` | `B`, `Q` | normalize over `K` | `[B, Q, K]` |
| `Linear` on `[B, T, In]` | `B`, `T` | project `In -> Out` | `[B, T, Out]` |


In [ ]:
{
// Tensor Dim Roles

function show(label: string, value: { shape: readonly number[], to_array(): unknown }) {
  console.log(label)
  console.log('shape:', value.shape)
  console.log(JSON.stringify(value.to_array()))
}


const x = tensor([
  [[1, 2, 3, 4], [10, 20, 30, 40], [100, 200, 300, 400]],
  [[5, 6, 7, 8], [50, 60, 70, 80], [500, 600, 700, 800]],
])

const w = tensor([
  [1, 0],
  [0, 1],
  [1, 1],
  [2, -1],
])

show('x: [batch=2, m=3, k=4]', x)
show('w: [k=4, n=2]', w)
show('matmul(x, w): [batch=2, m=3, n=2]', matmul(x, w))


const bias = tensor([0.5, -0.5])
const projected = add(matmul(x, w), bias)

show('bias: [out_features=2]', bias)
show('add(matmul(x, w), bias): [batch=2, time=3, out_features=2]', projected)


show('x: [batch=2, time=3, hidden=4]', x)
show('mean(x, 2): [batch=2, time=3]', mean(x, 2))


show('mean(x, 2, true): [batch=2, time=3, 1]', mean(x, 2, true))


const scores = tensor([
  [[1, 2, 3], [2, 0, -1]],
  [[0, 0, 1], [3, 1, 0]],
])

show('scores: [batch=2, query_pos=2, key_pos=3]', scores)
show('softmax(scores, 2): same shape, normalized over key_pos', softmax(scores, 2))
}


## Tensor Creation And Metadata



## Tensor From Nested Arrays

Nested arrays define axes.

```txt
[[1, 2], [3, 4]] -> shape [2, 2]
```

<svg viewBox="0 0 720 185" width="720" height="185" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace}.cell{fill:#dbeafe;stroke:#60a5fa}.meta{fill:#ffedd5;stroke:#fb923c}</style>
  <text class="txt" x="24" y="28">Two outer rows, two values per row.</text>
  <rect class="cell" x="80" y="65" width="55" height="40"/><text class="mono" x="102" y="90">1</text>
  <rect class="cell" x="135" y="65" width="55" height="40"/><text class="mono" x="157" y="90">2</text>
  <rect class="cell" x="80" y="105" width="55" height="40"/><text class="mono" x="102" y="130">3</text>
  <rect class="cell" x="135" y="105" width="55" height="40"/><text class="mono" x="157" y="130">4</text>
  <rect class="meta" x="300" y="75" width="260" height="58" rx="4"/><text class="mono" x="325" y="100">shape [2, 2]</text><text class="mono" x="325" y="123">dtype f32, device cpu</text>
</svg>


## Fill Constructors

`empty` creates storage without a meaningful fill value. `zeros`, `ones`, and `full` create tensors from shape plus a fill rule.


## Random Constructors And Seed

`rand` samples from a uniform distribution. `randn` samples from a normal distribution. `seed` makes examples reproducible.

```txt
seed(1); rand([2])  -> repeatable random values
seed(1); randn([2]) -> repeatable normal values
```


## Number Lines

`arange` counts by step. `linspace` divides an interval into evenly spaced values.


## DType, Device, Layout, And Finite Checks

`cast` changes dtype. `move` changes device. `contiguous` materializes a contiguous layout. Finite helpers summarize whether values are usable for training.


In [ ]:
{
// Tensor Creation And Metadata

function show(label: string, value: { shape: readonly number[], dtype: string, device: string, to_array(): unknown }) {
  console.log(label)
  console.log('shape:', value.shape)
  console.log('dtype:', value.dtype)
  console.log('device:', value.device)
  console.log(JSON.stringify(value.to_array()))
}


const x = tensor([[1, 2], [3, 4]])
show('tensor([[1, 2], [3, 4]])', x)


const scratch = empty([2, 3])
console.log('empty([2, 3]) shape:', scratch.shape, 'dtype:', scratch.dtype, 'device:', scratch.device)
show('zeros([2, 3])', zeros([2, 3]))
show('ones([2, 3])', ones([2, 3]))
show('full([2, 2], 7)', full([2, 2], 7))


seed(1)
show('rand([2])', rand([2]))
seed(1)
show('randn([2])', randn([2]))


show('arange(0, 5, 2)', arange(0, 5, 2))
show('linspace(0, 1, 5)', linspace(0, 1, 5))


show('cast(x, f64)', cast(x, 'f64'))
show('move(x, cpu)', move(x, 'cpu'))
show('contiguous(x)', contiguous(x))
console.log('finite_summary([1, Infinity, NaN]):', JSON.stringify(finite_summary(tensor([1, Infinity, NaN]))))
console.log('finite_abs_max([-3, 2]):', finite_abs_max(tensor([-3, 2])))
}


## Elementwise Ops



## Arithmetic

$$
z_{i,j} = a_{i,j} + b_{i,j}
$$

The same idea applies to `sub`, `mul`, and `div`: matching positions are combined.

<svg viewBox="0 0 760 245" width="760" height="245" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace;fill:#0f172a}.cellA{fill:#dbeafe;stroke:#60a5fa}.cellB{fill:#dcfce7;stroke:#4ade80}.cellC{fill:#ffedd5;stroke:#fb923c}.op{font:22px monospace;fill:#334155}</style>
  <text class="txt" x="24" y="28">Each output cell comes from the cells at the same position.</text>
  <rect class="cellA" x="70" y="62" width="54" height="40"/><text class="mono" x="90" y="87">-2</text>
  <rect class="cellA" x="124" y="62" width="54" height="40"/><text class="mono" x="146" y="87">-1</text>
  <rect class="cellA" x="178" y="62" width="54" height="40"/><text class="mono" x="199" y="87">0</text>
  <rect class="cellA" x="70" y="102" width="54" height="40"/><text class="mono" x="93" y="127">1</text>
  <rect class="cellA" x="124" y="102" width="54" height="40"/><text class="mono" x="147" y="127">2</text>
  <rect class="cellA" x="178" y="102" width="54" height="40"/><text class="mono" x="201" y="127">3</text>
  <text class="op" x="258" y="109">+</text>
  <rect class="cellB" x="300" y="62" width="54" height="40"/><text class="mono" x="318" y="87">10</text>
  <rect class="cellB" x="354" y="62" width="54" height="40"/><text class="mono" x="372" y="87">10</text>
  <rect class="cellB" x="408" y="62" width="54" height="40"/><text class="mono" x="426" y="87">10</text>
  <rect class="cellB" x="300" y="102" width="54" height="40"/><text class="mono" x="318" y="127">20</text>
  <rect class="cellB" x="354" y="102" width="54" height="40"/><text class="mono" x="372" y="127">20</text>
  <rect class="cellB" x="408" y="102" width="54" height="40"/><text class="mono" x="426" y="127">20</text>
  <text class="op" x="488" y="109">=</text>
  <rect class="cellC" x="530" y="62" width="54" height="40"/><text class="mono" x="550" y="87">8</text>
  <rect class="cellC" x="584" y="62" width="54" height="40"/><text class="mono" x="604" y="87">9</text>
  <rect class="cellC" x="638" y="62" width="54" height="40"/><text class="mono" x="653" y="87">10</text>
  <rect class="cellC" x="530" y="102" width="54" height="40"/><text class="mono" x="545" y="127">21</text>
  <rect class="cellC" x="584" y="102" width="54" height="40"/><text class="mono" x="599" y="127">22</text>
  <rect class="cellC" x="638" y="102" width="54" height="40"/><text class="mono" x="653" y="127">23</text>
  <text class="mono" x="70" y="185">result[0,0] = a[0,0] + b[0,0] = -2 + 10 = 8</text>
</svg>


## Unary Math

Unary ops use one input tensor. Each output value depends on one input value.

| Op | Rule |
| --- | --- |
| `neg` | $z = -x$ |
| `abs` | $z = |x|$ |
| `sign` | negative -> `-1`, zero -> `0`, positive -> `1` |
| `square` | $z = x^2$ |
| `sqrt` | $z = \sqrt{x}$ |
| `exp` | $z = e^x$ |
| `log` | $z = \log(x)$ |

<svg viewBox="0 0 760 210" width="760" height="210" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace;fill:#0f172a}.in{fill:#dbeafe;stroke:#60a5fa}.out{fill:#ffedd5;stroke:#fb923c}.arrow{stroke:#475569;stroke-width:2;marker-end:url(#arrow)}</style>
  <defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto"><path d="M0,0 L0,6 L9,3 z" fill="#475569"/></marker></defs>
  <text class="txt" x="24" y="28">Unary rule: one input number becomes one output number.</text>
  <rect class="in" x="80" y="70" width="84" height="48" rx="4"/><text class="mono" x="102" y="100">-3</text>
  <line class="arrow" x1="180" y1="94" x2="310" y2="94"/><text class="mono" x="214" y="78">abs(x)</text>
  <rect class="out" x="330" y="70" width="84" height="48" rx="4"/><text class="mono" x="362" y="100">3</text>
  <rect class="in" x="470" y="70" width="84" height="48" rx="4"/><text class="mono" x="502" y="100">3</text>
  <line class="arrow" x1="570" y1="94" x2="700" y2="94"/><text class="mono" x="602" y="78">square(x)</text>
  <rect class="out" x="710" y="70" width="42" height="48" rx="4"/><text class="mono" x="723" y="100">9</text>
  <text class="txt" x="80" y="160">No neighboring values are inspected.</text>
</svg>


## Clamp

$$
\mathrm{clamp}(x, min, max) =
\begin{cases}
min & x < min \\
x & min \le x \le max \\
max & x > max
\end{cases}
$$

Clamp is a value fence: values below the lower fence move up, values above the upper fence move down.

<svg viewBox="0 0 760 210" width="760" height="210" xmlns="http://www.w3.org/2000/svg">
  <style>.axis{stroke:#475569;stroke-width:2}.fence{stroke:#ef4444;stroke-width:3;stroke-dasharray:6 5}.pt{fill:#2563eb}.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace;fill:#0f172a}</style>
  <text class="txt" x="24" y="28">Clamp to the interval [-1, 1].</text>
  <line class="axis" x1="100" y1="105" x2="660" y2="105"/>
  <line class="fence" x1="270" y1="65" x2="270" y2="145"/><line class="fence" x1="490" y1="65" x2="490" y2="145"/>
  <text class="mono" x="258" y="165">-1</text><text class="mono" x="484" y="165">1</text>
  <circle class="pt" cx="160" cy="105" r="7"/><text class="mono" x="138" y="88">-3 -> -1</text>
  <circle class="pt" cx="380" cy="105" r="7"/><text class="mono" x="356" y="88">0 -> 0</text>
  <circle class="pt" cx="610" cy="105" r="7"/><text class="mono" x="590" y="88">3 -> 1</text>
</svg>


## Conditions And Masks

`gt_scalar(x, threshold)` creates a mask. `where(mask, a, b)` and `masked_fill(input, mask, value)` use that mask position by position.

$$
\mathrm{where}(cond, a, b)_{i,j} =
\begin{cases}
a_{i,j} & cond_{i,j} \\
b_{i,j} & \text{otherwise}
\end{cases}
$$

<svg viewBox="0 0 760 245" width="760" height="245" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace;fill:#0f172a}.m0{fill:#f1f5f9;stroke:#94a3b8}.m1{fill:#dcfce7;stroke:#4ade80}.out{fill:#ffedd5;stroke:#fb923c}.op{font:22px monospace;fill:#334155}</style>
  <text class="txt" x="24" y="28">The mask decides which positions are changed.</text>
  <rect class="m0" x="70" y="62" width="54" height="40"/><text class="mono" x="94" y="87">0</text>
  <rect class="m0" x="124" y="62" width="54" height="40"/><text class="mono" x="148" y="87">0</text>
  <rect class="m0" x="178" y="62" width="54" height="40"/><text class="mono" x="202" y="87">0</text>
  <rect class="m1" x="70" y="102" width="54" height="40"/><text class="mono" x="94" y="127">1</text>
  <rect class="m1" x="124" y="102" width="54" height="40"/><text class="mono" x="148" y="127">1</text>
  <rect class="m1" x="178" y="102" width="54" height="40"/><text class="mono" x="202" y="127">1</text>
  <text class="op" x="260" y="110">mask</text>
  <text class="op" x="355" y="110">+</text>
  <text class="mono" x="400" y="90">fill true cells</text>
  <text class="op" x="545" y="110">=</text>
  <rect class="out" x="590" y="62" width="54" height="40"/><text class="mono" x="603" y="87">-2</text>
  <rect class="out" x="644" y="62" width="54" height="40"/><text class="mono" x="657" y="87">-1</text>
  <rect class="out" x="698" y="62" width="54" height="40"/><text class="mono" x="722" y="87">0</text>
  <rect class="out" x="590" y="102" width="54" height="40"/><text class="mono" x="606" y="127">99</text>
  <rect class="out" x="644" y="102" width="54" height="40"/><text class="mono" x="660" y="127">99</text>
  <rect class="out" x="698" y="102" width="54" height="40"/><text class="mono" x="714" y="127">99</text>
  <text class="mono" x="70" y="188">gt_scalar(a, 0): true where a is greater than 0</text>
</svg>


## High-rank reading rule

For elementwise ops, all dimensions are parallel:

```txt
[batch, time, hidden] -> [batch, time, hidden]
```

Meaning:

```txt
For every batch item,
for every time position,
for every hidden value,
apply the same scalar rule.
```

Common mistakes:

- Expecting elementwise ops to mix values across a dimension. They do not.
- Forgetting that `log` and `sqrt` need valid input ranges.
- Treating masks as ordinary labels; masks control positions.


In [ ]:
{
// Elementwise Ops

function show(label: string, value: { shape: readonly number[], to_array(): unknown }) {
  console.log(label)
  console.log('shape:', value.shape)
  console.log(JSON.stringify(value.to_array()))
}

const a = tensor([[-2, -1, 0], [1, 2, 3]])
const b = tensor([[10, 10, 10], [20, 20, 20]])

show('a', a)
show('b', b)


show('add(a, b)', add(a, b))
show('sub(b, a)', sub(b, a))
show('mul(a, b)', mul(a, b))
show('div(b, tensor([[2, 5, 10], [4, 5, 10]]))', div(b, tensor([[2, 5, 10], [4, 5, 10]])))


show('neg(a)', neg(a))
show('abs(a)', abs(a))
show('sign(a)', sign(a))
show('square(a)', square(a))
show('sqrt(tensor([[1, 4, 9], [16, 25, 36]]))', sqrt(tensor([[1, 4, 9], [16, 25, 36]])))
show('exp(tensor([-1, 0, 1]))', exp(tensor([-1, 0, 1])))
show('log(tensor([1, 2, 4]))', log(tensor([1, 2, 4])))


show('clamp(a, -1, 1)', clamp(a, -1, 1))


const positive = gt_scalar(a, 0)
show('gt_scalar(a, 0)', positive)
show('where(gt_scalar(a, 0), b, a)', where(positive, b, a))
show('masked_fill(a, gt_scalar(a, 0), 99)', masked_fill(a, positive, 99))
}


## Activations



## ReLU

$$
\mathrm{ReLU}(x) = \max(x, 0)
$$

ReLU keeps positive values and clips negative values to zero.

<svg viewBox="0 0 760 250" width="760" height="250" xmlns="http://www.w3.org/2000/svg">
  <style>.axis{stroke:#475569;stroke-width:1}.grid{stroke:#e2e8f0;stroke-width:1}.curve{fill:none;stroke:#2563eb;stroke-width:4;stroke-linecap:round;stroke-linejoin:round}.txt{font:14px sans-serif;fill:#0f172a}.math{font:15px monospace;fill:#0f172a}.neg{fill:#fee2e2;stroke:#f87171}.pos{fill:#dcfce7;stroke:#4ade80}</style>
  <text class="txt" x="24" y="28">Negative side becomes zero; positive side passes through.</text>
  <line class="grid" x1="70" y1="180" x2="350" y2="180"/><line class="grid" x1="210" y1="50" x2="210" y2="210"/>
  <line class="axis" x1="70" y1="180" x2="350" y2="180"/><line class="axis" x1="210" y1="210" x2="210" y2="50"/>
  <path class="curve" d="M90 180 L210 180 L335 60"/>
  <text class="txt" x="342" y="185">x</text><text class="txt" x="218" y="58">y</text><text class="txt" x="198" y="198">0</text>
  <rect class="neg" x="430" y="70" width="105" height="42" rx="4"/><text class="math" x="448" y="96">-2 -> 0</text>
  <rect class="neg" x="430" y="124" width="105" height="42" rx="4"/><text class="math" x="448" y="150">-1 -> 0</text>
  <rect class="pos" x="555" y="70" width="105" height="42" rx="4"/><text class="math" x="573" y="96">1 -> 1</text>
  <rect class="pos" x="555" y="124" width="105" height="42" rx="4"/><text class="math" x="573" y="150">2 -> 2</text>
</svg>


## Sigmoid

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

Sigmoid squeezes any number into the range `(0, 1)`. It is often used when a value should behave like a probability-like score.

<svg viewBox="0 0 760 250" width="760" height="250" xmlns="http://www.w3.org/2000/svg">
  <style>.axis{stroke:#475569;stroke-width:1}.grid{stroke:#e2e8f0;stroke-width:1}.curve{fill:none;stroke:#2563eb;stroke-width:4;stroke-linecap:round}.txt{font:14px sans-serif;fill:#0f172a}.math{font:15px monospace;fill:#0f172a}.band{fill:#eff6ff;stroke:#93c5fd}</style>
  <text class="txt" x="24" y="28">Large negative values move toward 0; large positive values move toward 1.</text>
  <line class="grid" x1="70" y1="180" x2="350" y2="180"/><line class="grid" x1="70" y1="70" x2="350" y2="70"/><line class="grid" x1="210" y1="50" x2="210" y2="210"/>
  <line class="axis" x1="70" y1="180" x2="350" y2="180"/><line class="axis" x1="210" y1="210" x2="210" y2="50"/>
  <path class="curve" d="M90 172 C135 170, 170 150, 210 125 C250 100, 285 80, 335 73"/>
  <text class="txt" x="355" y="185">x</text><text class="txt" x="218" y="58">y</text><text class="txt" x="54" y="74">1</text><text class="txt" x="54" y="184">0</text><text class="txt" x="214" y="142">0.5</text>
  <rect class="band" x="430" y="72" width="220" height="42" rx="4"/><text class="math" x="448" y="98">sigmoid(0) = 0.5</text>
  <rect class="band" x="430" y="126" width="220" height="42" rx="4"/><text class="math" x="448" y="152">output is always 0..1</text>
</svg>


## Tanh

$$
\tanh(x) = \frac{e^{2x} - 1}{e^{2x} + 1}
$$

Tanh squeezes values into `(-1, 1)` and is centered at zero.

<svg viewBox="0 0 760 250" width="760" height="250" xmlns="http://www.w3.org/2000/svg">
  <style>.axis{stroke:#475569;stroke-width:1}.grid{stroke:#e2e8f0;stroke-width:1}.curve{fill:none;stroke:#2563eb;stroke-width:4;stroke-linecap:round}.txt{font:14px sans-serif;fill:#0f172a}.math{font:15px monospace;fill:#0f172a}.band{fill:#f0fdf4;stroke:#86efac}</style>
  <text class="txt" x="24" y="28">Like sigmoid, but centered: negative inputs stay negative, positive inputs stay positive.</text>
  <line class="grid" x1="70" y1="180" x2="350" y2="180"/><line class="grid" x1="70" y1="70" x2="350" y2="70"/><line class="grid" x1="70" y1="125" x2="350" y2="125"/><line class="grid" x1="210" y1="50" x2="210" y2="210"/>
  <line class="axis" x1="70" y1="125" x2="350" y2="125"/><line class="axis" x1="210" y1="210" x2="210" y2="50"/>
  <path class="curve" d="M90 178 C135 176, 170 155, 210 125 C250 95, 285 74, 335 72"/>
  <text class="txt" x="355" y="130">x</text><text class="txt" x="218" y="58">y</text><text class="txt" x="52" y="74">1</text><text class="txt" x="47" y="184">-1</text><text class="txt" x="214" y="142">0</text>
  <rect class="band" x="430" y="72" width="230" height="42" rx="4"/><text class="math" x="448" y="98">tanh(0) = 0</text>
  <rect class="band" x="430" y="126" width="230" height="42" rx="4"/><text class="math" x="448" y="152">output is always -1..1</text>
</svg>


## GELU

$$
\mathrm{GELU}(x) = \frac{1}{2}x\left(1+\tanh\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right)
$$

GELU is a smooth gate. Negative values are mostly suppressed, but not with a hard corner like ReLU.

<svg viewBox="0 0 760 250" width="760" height="250" xmlns="http://www.w3.org/2000/svg">
  <style>.axis{stroke:#475569;stroke-width:1}.grid{stroke:#e2e8f0;stroke-width:1}.curve{fill:none;stroke:#2563eb;stroke-width:4;stroke-linecap:round;stroke-linejoin:round}.relu{fill:none;stroke:#94a3b8;stroke-width:2;stroke-dasharray:6 5}.txt{font:14px sans-serif;fill:#0f172a}.math{font:15px monospace;fill:#0f172a}.band{fill:#f8fafc;stroke:#cbd5e1}</style>
  <text class="txt" x="24" y="28">GELU behaves like a soft, smooth version of ReLU.</text>
  <line class="grid" x1="70" y1="180" x2="350" y2="180"/><line class="grid" x1="210" y1="50" x2="210" y2="210"/>
  <line class="axis" x1="70" y1="180" x2="350" y2="180"/><line class="axis" x1="210" y1="210" x2="210" y2="50"/>
  <path class="relu" d="M90 180 L210 180 L335 60"/>
  <path class="curve" d="M90 181 C135 182, 170 190, 195 187 C220 183, 238 160, 260 132 C285 100, 310 75, 335 60"/>
  <text class="txt" x="355" y="185">x</text><text class="txt" x="218" y="58">y</text><text class="txt" x="238" y="206">gray = ReLU reference</text>
  <rect class="band" x="430" y="72" width="260" height="42" rx="4"/><text class="math" x="448" y="98">GELU(-1) is small negative</text>
  <rect class="band" x="430" y="126" width="260" height="42" rx="4"/><text class="math" x="448" y="152">GELU(2) is almost 2</text>
</svg>


## SiLU

$$
\mathrm{SiLU}(x) = x \cdot \sigma(x)
$$

SiLU multiplies the input by its sigmoid gate. It keeps large positive values and smoothly dampens negative values.

<svg viewBox="0 0 760 250" width="760" height="250" xmlns="http://www.w3.org/2000/svg">
  <style>.axis{stroke:#475569;stroke-width:1}.grid{stroke:#e2e8f0;stroke-width:1}.curve{fill:none;stroke:#2563eb;stroke-width:4;stroke-linecap:round;stroke-linejoin:round}.txt{font:14px sans-serif;fill:#0f172a}.math{font:15px monospace;fill:#0f172a}.gate{fill:#eff6ff;stroke:#93c5fd}</style>
  <text class="txt" x="24" y="28">SiLU is input times a sigmoid gate.</text>
  <line class="grid" x1="70" y1="180" x2="350" y2="180"/><line class="grid" x1="210" y1="50" x2="210" y2="210"/>
  <line class="axis" x1="70" y1="180" x2="350" y2="180"/><line class="axis" x1="210" y1="210" x2="210" y2="50"/>
  <path class="curve" d="M90 183 C125 186, 160 193, 190 190 C220 186, 242 164, 265 132 C290 98, 315 72, 335 60"/>
  <text class="txt" x="355" y="185">x</text><text class="txt" x="218" y="58">y</text>
  <rect class="gate" x="430" y="60" width="94" height="38" rx="4"/><text class="math" x="455" y="84">x</text>
  <text class="math" x="540" y="84">*</text>
  <rect class="gate" x="565" y="60" width="94" height="38" rx="4"/><text class="math" x="581" y="84">sigmoid(x)</text>
  <text class="math" x="500" y="142">negative: dampened</text>
  <text class="math" x="500" y="170">positive: mostly kept</text>
</svg>


## High-rank reading rule

Because these activations are elementwise, all dimensions are parallel dimensions.

```txt
[batch, time, hidden] -> [batch, time, hidden]
```

Meaning:

```txt
For every batch item,
for every time position,
for every hidden value,
apply the same scalar activation formula.
```


In [ ]:
{
// Activations

function show(label: string, value: { shape: readonly number[], to_array(): unknown }) {
  console.log(label)
  console.log('shape:', value.shape)
  console.log(JSON.stringify(value.to_array()))
}

const xs = tensor([-4, -2, -1, 0, 1, 2, 4])
show('input xs', xs)


show('relu(xs)', relu(xs))


show('sigmoid(xs)', sigmoid(xs))


show('tanh(xs)', tanh(xs))


show('gelu(xs)', gelu(xs))


show('silu(xs)', silu(xs))
}


## Reductions And Softmax



## Sum And Mean

$$
\mathrm{sum}(x, dim) = \sum_i x_i
$$

$$
\mathrm{mean}(x, dim) = \frac{1}{N}\sum_i x_i
$$

<svg viewBox="0 0 760 210" width="760" height="210" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace}.p{fill:#dbeafe;stroke:#60a5fa}.r{fill:#fee2e2;stroke:#f87171}.o{fill:#ffedd5;stroke:#fb923c}</style>
  <text class="txt" x="24" y="28">For each parallel position, reduce the hidden values.</text>
  <rect class="p" x="70" y="65" width="80" height="40" rx="4"/><text class="mono" x="91" y="90">batch</text>
  <rect class="p" x="155" y="65" width="70" height="40" rx="4"/><text class="mono" x="176" y="90">time</text>
  <rect class="r" x="230" y="65" width="80" height="40" rx="4"/><text class="mono" x="248" y="90">hidden</text>
  <text class="mono" x="335" y="90">mean(dim=2)</text>
  <text class="mono" x="475" y="90">-></text>
  <rect class="p" x="520" y="65" width="80" height="40" rx="4"/><text class="mono" x="541" y="90">batch</text>
  <rect class="p" x="605" y="65" width="70" height="40" rx="4"/><text class="mono" x="626" y="90">time</text>
  <text class="mono" x="70" y="150">x[0,0] = [1,2,3], sum = 6, mean = 2</text>
</svg>


### Before You Run

For `x: [batch=2, time=2, hidden=3]`, predict the shape of `sum(x, 2)` and `mean(x, 2, true)`.


## Extremes And Positions

`max` and `min` return values. `argmax` and `argmin` return positions.

```txt
[4, 5, 6] -> max = 6, argmax = 2
```


## Variance And Std

Variance and standard deviation measure spread around the mean.

$$
\mathrm{variance}(x)=\frac{1}{N}\sum_i(x_i-\bar{x})^2
$$

$$
\mathrm{std}(x)=\sqrt{\mathrm{variance}(x)}
$$


## Softmax

$$
\mathrm{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}
$$

Softmax normalizes one dimension and preserves shape.

```txt
[batch, query_pos, key_pos] -- softmax(dim=2) --> same shape
parallel: batch, query_pos
normalized: key_pos
```


### Read The Output

After `softmax(scores, 2)`, each length-3 vector along `key_pos` sums to about 1. The shape is unchanged because softmax normalizes an axis instead of reducing it.


In [ ]:
{
// Reductions And Softmax

function show(label: string, value: { shape: readonly number[], to_array(): unknown }) {
  console.log(label)
  console.log('shape:', value.shape)
  console.log(JSON.stringify(value.to_array()))
}

const x = tensor([
  [[1, 2, 3], [4, 5, 6]],
  [[10, 20, 30], [40, 50, 60]],
])
show('x: [batch=2, time=2, hidden=3]', x)


show('sum(x, 2): [batch=2, time=2]', sum(x, 2))
show('mean(x, 2): [batch=2, time=2]', mean(x, 2))
show('mean(x, 2, true): [batch=2, time=2, 1]', mean(x, 2, true))


show('max(x, 2)', max(x, 2))
show('min(x, 2)', min(x, 2))
show('argmax(x, 2)', argmax(x, 2))
show('argmin(x, 2)', argmin(x, 2))


show('variance(x, 2)', variance(x, 2))
show('std(x, 2)', std(x, 2))


const scores = tensor([[[1, 2, 3], [2, 0, -1]]])
show('scores: [batch=1, query_pos=2, key_pos=3]', scores)
show('softmax(scores, 2)', softmax(scores, 2))
}


## Shape And Selection



## Reshape, Transpose, Permute

`reshape` keeps the same values but groups them into new axes.

`transpose` swaps two axes. `permute` supplies the full axis order.

<svg viewBox="0 0 760 200" width="760" height="200" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace}.a{fill:#dbeafe;stroke:#60a5fa}.b{fill:#ffedd5;stroke:#fb923c}</style>
  <text class="txt" x="24" y="28">Shape changes are often address changes, not new math.</text>
  <rect class="a" x="80" y="70" width="90" height="42" rx="4"/><text class="mono" x="103" y="96">[2,3]</text>
  <text class="mono" x="210" y="96">reshape</text>
  <rect class="b" x="320" y="70" width="90" height="42" rx="4"/><text class="mono" x="343" y="96">[3,2]</text>
  <text class="mono" x="455" y="96">transpose</text>
  <rect class="b" x="590" y="70" width="90" height="42" rx="4"/><text class="mono" x="613" y="96">[3,2]</text>
</svg>


## Squeeze And Unsqueeze

`unsqueeze` inserts a size-1 axis. `squeeze` removes size-1 axes.

```txt
[2, 3] -- unsqueeze(0) --> [1, 2, 3]
[1, 2, 3] -- squeeze(0) --> [2, 3]
```


## Cat And Stack

`cat` extends an existing axis. `stack` creates a new axis.

```txt
cat([x, x], dim=0):   [2,3] + [2,3] -> [4,3]
stack([x, x], dim=0): [2,3], [2,3] -> [2,2,3]
```


## Slice, Gather, Index Select, TopK, One Hot

Selection ops answer: which positions do you want?

- `slice`: choose ranges or fixed positions.
- `index_select`: choose items along one axis.
- `gather`: choose per-position indices from an index tensor.
- `topk`: choose the largest `k` values and their positions.
- `one_hot`: turn class ids into indicator rows.


## Common Mistakes

- Confusing `cat` and `stack`: `stack` adds a new axis.
- Forgetting that `reshape` must keep the same total number of elements.
- Using `gather` when `index_select` is enough; `gather` is for per-position indices.


In [ ]:
{
// Shape And Selection

function show(label: string, value: { shape: readonly number[], to_array(): unknown }) {
  console.log(label)
  console.log('shape:', value.shape)
  console.log(JSON.stringify(value.to_array()))
}

const x = tensor([[1, 2, 3], [4, 5, 6]])
show('x: [2, 3]', x)


show('reshape(x, [3, 2])', reshape(x, [3, 2]))
show('transpose(x, 0, 1)', transpose(x, 0, 1))
const y = tensor([[[1, 2], [3, 4], [5, 6]]])
show('permute(y, [1, 0, 2])', permute(y, [1, 0, 2]))


const withBatch = unsqueeze(x, 0)
show('unsqueeze(x, 0)', withBatch)
show('squeeze(withBatch, 0)', squeeze(withBatch, 0))


show('cat([x, x], 0)', cat([x, x], 0))
show('cat([x, x], 1)', cat([x, x], 1))
show('stack([x, x], 0)', stack([x, x], 0))


show('slice(x, range(0, 2), 1)', slice(x, range(0, 2), 1))
show('index_select(x, 0, tensor([1, 0]))', index_select(x, 0, tensor([1, 0], { dtype: 'i64' })))
show('gather(x, 1, [[2,1],[0,2]])', gather(x, 1, tensor([[2, 1], [0, 2]], { dtype: 'i64' })))
const tk = topk(x, 2, 1)
show('topk(x, 2, 1).values', tk.values)
show('topk(x, 2, 1).indices', tk.indices)
show('one_hot([0, 2, 1], 3)', one_hot(tensor([0, 2, 1], { dtype: 'i64' }), 3))
}
